# V8 — 02 Features: shrinkage, time decay, xG proxy

**Purpose.** Show that the Phase 2 features behave the way the methodology says
they do. Three claims, each checked here rather than asserted:

1. **Shrinkage** pulls thin-sample teams toward the league average, and releases
   them as evidence accumulates.
2. **Time decay** makes recent matches matter more, by a known amount.
3. **Strengths** end up ranking teams in a way that matches reality.

The first section is a toy example with numbers you can verify by hand. That is
the Phase 2 "done when" criterion: *the shrinkage behaviour is demonstrably
correct on a toy example.*

Restart and Run All before committing.


In [ ]:
# --- Setup -------------------------------------------------------------------
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from src.config import load_config
from src import features, ingest

pd.set_option("display.width", 120)
config = load_config(REPO_ROOT / "config.yaml")

K = config["features"]["shrinkage_k"]
HALF_LIFE = config["features"]["half_life_days"]
print(f"shrinkage k = {K} | time-decay half-life = {HALF_LIFE} days")

## 1. Shrinkage on a toy example

A team scores at **twice** the league average — but only in a handful of
matches. How much should we believe it?

$$\hat\theta^{\text{shrunk}} = w\,\hat\theta + (1-w)\,\bar\theta,
\qquad w = \frac{n}{n+k}$$

With $k = 8$, a team with 8 matches gets $w = 0.5$ — exactly halfway between its
own record and the league average. Check the table against that by hand.

In [ ]:
# --- One team, the same record, different amounts of evidence ----------------
raw_estimate = 2.0          # scoring at twice the league average
league_average = 1.0

rows = []
for n in [0, 1, 2, 4, 8, 16, 32, 64]:
    shrunk = features.shrink(raw_estimate, n_matches=n, prior=league_average, k=K)
    rows.append({
        "matches_played": n,
        "weight_on_own_record": n / (n + K),
        "raw_estimate": raw_estimate,
        "shrunk_estimate": shrunk,
    })

table = pd.DataFrame(rows)
print(table.round(3).to_string(index=False))

print(f"\nSanity checks:")
print(f"  n=0  -> {table.loc[0, 'shrunk_estimate']:.3f}  (must be exactly the prior, 1.0)")
print(f"  n={K} -> {table.loc[table['matches_played'] == K, 'shrunk_estimate'].iloc[0]:.3f}"
      f"  (must be exactly halfway, 1.5)")

In [ ]:
# --- Chart: evidence earns trust gradually -----------------------------------
n_grid = np.arange(0, 60)
fig, ax = plt.subplots(figsize=(7, 4))

for k in [2, K, 20]:
    curve = [features.shrink(2.0, n, prior=1.0, k=k) for n in n_grid]
    style = "-" if k == K else "--"
    ax.plot(n_grid, curve, style, label=f"k = {k}" + (" (ours)" if k == K else ""))

ax.axhline(2.0, color="grey", lw=1, label="team's own record")
ax.axhline(1.0, color="black", lw=1, label="league average (prior)")
ax.set_xlabel("matches played")
ax.set_ylabel("estimated attack strength")
ax.set_title("Shrinkage: how fast a team earns its own rating")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Read the chart.** There is no cliff. A larger $k$ is a more sceptical prior:
the team has to play more matches before its own record dominates.

**The credit-risk translation,** worth having ready for an interview: this is
exactly a thin-file adjustment. A borrower with three months of history, or a
portfolio segment with four observed defaults, gets pulled toward the portfolio
average for the same reason and by the same formula. The quantity being shrunk
changes; the argument does not.

## 2. Time decay

A match one half-life old counts half as much as one played today:
$w = 0.5^{\,\Delta t / H}$.

In [ ]:
# --- The decay curve, with the checkable points marked -----------------------
days = np.arange(0, 730)
dates = pd.Timestamp("2026-01-01") - pd.to_timedelta(days, unit="D")
weights = features.time_weights(dates, as_of="2026-01-01", half_life_days=HALF_LIFE)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(days, weights, color="#2c7fb8")
for multiple in (1, 2, 3):
    age = HALF_LIFE * multiple
    ax.plot(age, 0.5 ** multiple, "o", color="black")
    ax.annotate(f"{age}d -> {0.5 ** multiple:.3f}", (age, 0.5 ** multiple),
                textcoords="offset points", xytext=(8, 8), fontsize=8)

ax.set_xlabel("match age (days)")
ax.set_ylabel("weight")
ax.set_title(f"Exponential time decay (half-life {HALF_LIFE} days)")
plt.tight_layout()
plt.show()

print("A match played today counts 1.000;"
      f" one a season old ({HALF_LIFE}d) counts {0.5:.3f};"
      f" two seasons old counts {0.25:.3f}.")

**What this buys.** Half a season of memory is a judgement call, not a law.
Too short and the model forgets what a team is; too long and it keeps rating a
squad that has since sold its striker. `features.half_life_days` is in config
precisely so Phase 3 can tune it rather than argue about it.

## 3. The real feature matrix

Everything above on actual data. Note the **warm-up rows**: early-season matches
where neither team has enough history to be trusted yet. They are flagged, not
deleted — Phase 3 decides whether to train on them.

In [ ]:
# --- Build features from the canonical table ---------------------------------
canonical_path = (REPO_ROOT / config["data"]["processed_dir"]
                  / config["validation"]["canonical_filename"])

if canonical_path.exists():
    matches = pd.read_csv(canonical_path, parse_dates=["MatchDate"])
    print(f"Loaded canonical table: {len(matches):,} matches")
else:
    raw_files = [f for f in ingest.expected_raw_files(config) if f["exists"]]
    if raw_files:
        raw = ingest.load_many(raw_files, config)
    else:
        raw = ingest.load_matches(REPO_ROOT / config["data"]["sample_path"], config)
        print("Using the bundled sample — numbers below are illustrative only.")
    matches, _ = ingest.run_ingest(raw, config, save=False)

feature_matrix, report = features.build_features(matches, config)
print()
print(features.feature_report(feature_matrix, report))

In [ ]:
# --- Where did each team end up? ---------------------------------------------
# Strengths as at the last match date: the model's final view of the division.
blended, note = features.blend_goals(matches, matches, config)
final = features.team_strengths(
    blended, as_of=matches["MatchDate"].max() + pd.Timedelta(days=1), config=config
)

ranking = final.sort_values("attack", ascending=False).round(3)
print("Team strengths at the end of the data (1.00 = league average):")
print(ranking.to_string())

print("\nSanity check — these should look like a plausible league table.")
print("If a team you know to be strong sits below 1.0 on attack, investigate")
print("before trusting anything in Phase 3.")

In [ ]:
# --- Chart: attack vs defence, the shape of a division -----------------------
fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.scatter(final["attack"], final["defence"], s=40, color="#2c7fb8")

for team, row in final.iterrows():
    ax.annotate(str(team)[:14], (row["attack"], row["defence"]),
                fontsize=7, textcoords="offset points", xytext=(4, 3))

ax.axvline(1.0, color="grey", lw=1, ls="--")
ax.axhline(1.0, color="grey", lw=1, ls="--")
ax.set_xlabel("attack  (higher = scores more)")
ax.set_ylabel("defence  (LOWER = concedes less)")
ax.invert_yaxis()                     # so "good" is up and to the right
ax.set_title("Team strengths: bottom-left quadrant is the strongest")
plt.tight_layout()
plt.show()

## 4. Proving there is no lookahead

The claim that makes the backtest meaningful: **a match's features depend only on
matches played before it.** Below, features are rebuilt on a truncated copy of
the data and compared with the full run.

This is the notebook version of `test_features_do_not_use_the_future`. It is
worth seeing it pass with your own eyes once, because this is the bug that is
invisible in every other output.

In [ ]:
# --- Rebuild on half the data and compare ------------------------------------
cutoff = np.sort(matches["MatchDate"].unique())[len(matches["MatchDate"].unique()) // 2]
truncated_input = matches[matches["MatchDate"] < cutoff]
truncated, _ = features.build_features(truncated_input, config)

strength_cols = ["HomeAttack", "HomeDefence", "AwayAttack", "AwayDefence"]
shared = set(truncated["MatchID"])

a = (feature_matrix[feature_matrix["MatchID"].isin(shared)]
     .set_index("MatchID").sort_index()[strength_cols])
b = truncated.set_index("MatchID").sort_index()[strength_cols]

largest_difference = (a - b).abs().to_numpy().max()
print(f"Matches compared: {len(a):,}")
print(f"Largest difference in any strength value: {largest_difference:.2e}")
print()
print("PASS — features are identical with and without the future."
      if largest_difference < 1e-9 else
      "FAIL — the future is leaking into past features. Stop and fix this.")

## 5. Findings to record

Copy into `PROGRESS.md`:

1. Matches, usable rows, and warm-up rows from the feature report.
2. The attack/defence range on usable rows — a spread of roughly 0.5 to 1.6 is
   normal for a top league; much wider suggests shrinkage is too weak.
3. The conversion rate in the xG note (goals per shot on target).
4. The top and bottom three teams by attack strength — do they match what you
   know about those seasons?
5. The lookahead check result.

Item 4 is the real test. The maths can be perfect and the ranking still wrong if
something upstream is broken, and you are the only person who can spot it.